# 🚀 FEDHEP POST-MIDTERM EXPERIMENTS — PART 1 OF 2
## 🌐 STUDY A (Idea A): Multi-Seed Peer Clusters ($K$) Sensitivity & Topology Sweep

> **Mục tiêu nghiên cứu**: Khảo sát sự ảnh hưởng của số lượng peer cluster $K \in \{1, 2, 3, 5, 8, 10, 15\}$ đến độ chính xác cá nhân hóa (personalized accuracy) và độ công bằng (bottom-10% fairness) dưới 3 mức độ non-IID khác nhau ($lpha \in \{0.05, 0.1, 0.5\}$) với 3 seeds thống kê để vẽ error bands.
>
> ⚡ **Chạy song song**: Notebook này chạy độc lập với **Part 2** (nghiên cứu về Attacks & Majority $q \ge 50\%$) để tối ưu thời gian chạy trên Kaggle GPU (tận dụng 2 session GPU cùng lúc).

In [ ]:
import os
import sys

# 1. Repository Setup
ROOT = '/kaggle/working/Topology-aware-FDL'
if not os.path.exists(ROOT):
    print('🚀 Cloning repository from GitHub...')
    !git clone https://github.com/nam200718/Topology-aware-FDL.git {ROOT}
else:
    print('🔄 Repository exists, pulling latest updates from main...')
    !cd {ROOT} && git pull origin main

%cd {ROOT}
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

# 2. DEDICATED OUTPUT DIRECTORY (Outside of repo, directly visible in Kaggle Output)
if os.path.exists('/kaggle/working'):
    OUT_DIR = '/kaggle/working/oreh_results_part1'
else:
    OUT_DIR = os.path.abspath('./oreh_results_part1')

SANITY_DIR = os.path.join(OUT_DIR, 'sanity')
EVAL_DIR = os.path.join(OUT_DIR, 'eval')
FIG_DIR = os.path.join(OUT_DIR, 'figures')

os.makedirs(SANITY_DIR, exist_ok=True)
os.makedirs(EVAL_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

print('✅ Working Directory:', os.getcwd())
print('📁 Dedicated Results Directory:', OUT_DIR)
print('📊 CSV Tables Directory:', EVAL_DIR)
print('📈 Figures Directory:', FIG_DIR)

In [ ]:
!pip -q install pydantic pyyaml torchvision pandas matplotlib seaborn scipy

In [ ]:
from src.data.dataset import _auto_link_dataset
data_dir = _auto_link_dataset('cifar100')
print('📂 CIFAR-100 Dataset linked at:', data_dir)

In [ ]:
# ==============================================================================
# Cell 4: 1-Round Pre-Flight Sanity Check
# ==============================================================================
from oreh.comparison_runner import OrehComparisonRunner
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Running 1-round pre-flight sanity check on CIFAR-100 using {device}...')

sanity_runner = OrehComparisonRunner(
    dataset='cifar100',
    num_rounds=1,
    num_clients=15,
    num_clusters=3,
    alpha=0.5,
    device=device,
    train_subset=1000,
    test_subset=500,
    data_dir=data_dir,
    output_dir=SANITY_DIR,
)

for k_test in [2, 3]:
    test_res = sanity_runner.run_single_experiment(
        defense_scope='both',
        num_clusters=k_test,
        attack_type='none',
        byzantine_rate=0.0,
        seed=42,
    )
    print(f'✅ Sanity Check K={k_test} PASSED! Acc: {test_res["final_test_acc"]}% | Latency: {test_res["total_defense_latency_ms"]} ms')

print('\n🚀 Pre-Flight Sanity Checks Successfully Completed! Ready for Study A sweep.')

## 🌐 STUDY A (Idea A): Multi-Seed Peer Clusters ($K$) Sensitivity Sweep
Khảo sát số lượng peer cluster $K \in \{1, 2, 3, 5, 8, 10, 15\}$ trên 3 mức độ non-IID:
- **Extreme Skew**: $lpha = 0.05$ (mỗi client chỉ nắm 1-2 classes chủ đạo)
- **Severe Skew**: $lpha = 0.1$ (độ lệch nhãn cao)
- **Moderate Skew**: $lpha = 0.5$ (chuẩn benchmark phổ biến)

* **Seeds**: $\{42, 123, 7\}$ (cho phép vẽ standard deviation bands)
* **Output File**: `/kaggle/working/oreh_results_part1/eval/cifar100_postmid_k_clusters.csv` (ghi realtime sau mỗi lượt chạy)

In [ ]:
# ==============================================================================
# Cell 6: [STUDY A] Multi-Seed Peer Clusters (K) Sensitivity Sweep
# ==============================================================================
from oreh.comparison_runner import OrehComparisonRunner
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
runner_a = OrehComparisonRunner(
    dataset='cifar100',
    num_rounds=30,
    num_clients=15,
    device=device,
    train_subset=10000,
    test_subset=3000,
    data_dir=data_dir,
    output_dir=EVAL_DIR,
)

# Chạy sweep đa seeds: K in [1, 2, 3, 5, 8, 10, 15] across alphas [0.05, 0.1, 0.5]
df_k = runner_a.run_postmid_k_clusters(
    k_values=[1, 2, 3, 5, 8, 10, 15],
    alphas=[0.05, 0.1, 0.5],
    seeds=[42, 123, 7],
    byzantine_rate=0.0,
)

print('\n--- STUDY A (K CLUSTERS SWEEP) SUMMARY TABLE ---')
cols_k = ['num_clusters', 'alpha', 'seed', 'final_test_acc', 'bottom_10_fairness', 'total_defense_latency_ms']
if not df_k.empty:
    print(df_k[[c for c in cols_k if c in df_k.columns]].to_string(index=False))

## 📊 Publication-Quality Visualizations & Statistical Analysis
Vẽ biểu đồ tương quan giữa số cụm $K$ và độ chính xác trung bình (kèm shaded error band $\pm 1 	ext{ std}$) và lưu file hình ảnh độ phân giải cao vào thư mục `figures/`.

In [ ]:
# ==============================================================================
# Cell 8: Plotting Study A Results
# ==============================================================================
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

k_file = os.path.join(EVAL_DIR, 'cifar100_postmid_k_clusters.csv')
if os.path.exists(k_file):
    df_k = pd.read_csv(k_file)
    if not df_k.empty:
        # Plot 1: Accuracy vs K with Error Bands
        plt.figure(figsize=(9, 5.5))
        summary_k = df_k.groupby(['alpha', 'num_clusters'])['final_test_acc'].agg(['mean', 'std']).reset_index()
        colors = {'0.05': '#d62728', '0.1': '#ff7f0e', '0.5': '#1f77b4'}
        for a in sorted(summary_k['alpha'].unique()):
            sub = summary_k[summary_k['alpha'] == a].sort_values('num_clusters')
            k_vals = sub['num_clusters'].values
            means = sub['mean'].values
            stds = sub['std'].fillna(0).values
            c = colors.get(str(a), None)
            plt.plot(k_vals, means, marker='s', linewidth=2.5, label=f'Dirichlet $\\alpha = {a}$', color=c)
            plt.fill_between(k_vals, means - stds, means + stds, alpha=0.18, color=c)
            
        plt.title('CIFAR-100: Impact of Number of Peer Clusters ($K$) on Accuracy across Skews', fontsize=12, fontweight='bold')
        plt.xlabel('Number of Peer Clusters ($K$)', fontsize=11)
        plt.ylabel('Personalized Test Accuracy (%)', fontsize=11)
        plt.xticks([1, 2, 3, 5, 8, 10, 15])
        plt.grid(True, linestyle='--', alpha=0.6)
        plt.legend(frameon=True, fontsize=10)
        plt.tight_layout()
        acc_fig = os.path.join(FIG_DIR, 'postmid_a_k_cluster_multiseed_curve.png')
        plt.savefig(acc_fig, dpi=200)
        plt.show()
        print(f'✅ Saved accuracy plot to: {acc_fig}')

        # Plot 2: Bottom-10% Fairness vs K
        if 'bottom_10_fairness' in df_k.columns:
            plt.figure(figsize=(9, 5.5))
            fair_k = df_k.groupby(['alpha', 'num_clusters'])['bottom_10_fairness'].agg(['mean', 'std']).reset_index()
            for a in sorted(fair_k['alpha'].unique()):
                sub = fair_k[fair_k['alpha'] == a].sort_values('num_clusters')
                k_vals = sub['num_clusters'].values
                means = sub['mean'].values
                stds = sub['std'].fillna(0).values
                c = colors.get(str(a), None)
                plt.plot(k_vals, means, marker='o', linestyle='--', linewidth=2, label=f'Dirichlet $\\alpha = {a}$', color=c)
                plt.fill_between(k_vals, means - stds, means + stds, alpha=0.15, color=c)
            plt.title('CIFAR-100: Bottom-10% Fairness vs Peer Clusters ($K$)', fontsize=12, fontweight='bold')
            plt.xlabel('Number of Peer Clusters ($K$)', fontsize=11)
            plt.ylabel('Bottom-10% Client Accuracy (%)', fontsize=11)
            plt.xticks([1, 2, 3, 5, 8, 10, 15])
            plt.grid(True, linestyle='--', alpha=0.6)
            plt.legend(frameon=True, fontsize=10)
            plt.tight_layout()
            fair_fig = os.path.join(FIG_DIR, 'postmid_a_fairness_vs_k.png')
            plt.savefig(fair_fig, dpi=200)
            plt.show()
            print(f'✅ Saved fairness plot to: {fair_fig}')

## 📦 1-Click ZIP Packaging for Direct Download
Tự động nén toàn bộ thư mục `/kaggle/working/oreh_results_part1/` thành 1 file ZIP duy nhất. Bạn có thể download trực tiếp từ tab Output bên phải màn hình Kaggle mà không cần tìm kiếm trong repo code!

In [ ]:
# ==============================================================================
# Cell 10: 1-Click ZIP Packaging for Direct Download
# ==============================================================================
import shutil

zip_base = '/kaggle/working/oreh_part1_k_clusters_summary' if os.path.exists('/kaggle/working') else './oreh_part1_k_clusters_summary'
shutil.make_archive(zip_base, 'zip', OUT_DIR)

print('🎉 PART 1 EXPERIMENT & PACKAGING COMPLETED!')
print(f'📁 Dedicated Folder: {OUT_DIR}')
print(f'📦 Direct Download File: {zip_base}.zip')
print('You can download oreh_part1_k_clusters_summary.zip directly from the Kaggle Output sidebar!')